In [1]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

print(movies.shape, ratings.shape)
movies.head()

(9742, 3) (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [2]:
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()
print(f"Users: {n_users}, Movies rated: {n_movies}, Ratings: {len(ratings)}")

# How sparse is the data?
sparsity = 1 - len(ratings) / (n_users * n_movies)
print(f"Sparsity: {sparsity:.2%}")

# Rating distribution
ratings["rating"].value_counts().sort_index()

Users: 610, Movies rated: 9724, Ratings: 100836
Sparsity: 98.30%


rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [3]:
stats = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    n_ratings=("rating", "count")
).reset_index()

popular = (
    stats[stats["n_ratings"] >= 50]
    .merge(movies, on="movieId")
    .sort_values("avg_rating", ascending=False)
)

popular[["title", "genres", "avg_rating", "n_ratings"]].head(10)

,title,genres,avg_rating,n_ratings
53,"Shawshank Redemption, The (1994)",Crime|Drama,4.429022,317
118,"Godfather, The (1972)",Crime|Drama,4.289062,192
277,Fight Club (1999),Action|Crime|Drama|Thriller,4.272936,218
171,Cool Hand Luke (1967),Drama,4.271930,57
112,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.268041,97
120,Rear Window (1954),Mystery|Thriller,4.261905,84
158,"Godfather: Part II, The (1974)",Crime|Drama,4.259690,129
402,"Departed, The (2006)",Crime|Drama|Thriller,4.252336,107
153,Goodfellas (1990),Crime|Drama,4.250000,126
123,Casablanca (1942),Drama|Romance,4.240000,100


In [4]:
def top_popular(n=10, min_ratings=50, genre=None):
    df = stats[stats["n_ratings"] >= min_ratings].merge(movies, on="movieId")
    if genre:
        df = df[df["genres"].str.contains(genre, case=False)]
    return df.sort_values("avg_rating", ascending=False)[["title", "avg_rating", "n_ratings"]].head(n)

top_popular(5, genre="Comedy")

,title,avg_rating,n_ratings
112,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041,97
143,"Princess Bride, The (1987)",4.232394,142
49,Pulp Fiction (1994),4.197068,307
334,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",4.183333,120
62,Forrest Gump (1994),4.164134,329


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Clean genres into space-separated tokens (keep a new column, don't overwrite the original)
movies["genre_text"] = (
    movies["genres"]
    .str.replace("|", " ", regex=False)
    .str.replace("-", "", regex=False)          # Sci-Fi -> SciFi, Film-Noir -> FilmNoir
    .str.replace("(no genres listed)", "", regex=False)
)

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genre_text"])

print(genre_matrix.shape)
print(tfidf.get_feature_names_out())

(9742, 19)
['action' 'adventure' 'animation' 'children' 'comedy' 'crime'
 'documentary' 'drama' 'fantasy' 'filmnoir' 'horror' 'imax' 'musical'
 'mystery' 'romance' 'scifi' 'thriller' 'war' 'western']


In [6]:
sim = cosine_similarity(genre_matrix)
print(sim.shape)   # (9742, 9742)

(9742, 9742)


In [7]:
def similar_movies(title, n=10):
    matches = movies[movies["title"].str.contains(title, case=False, regex=False)]
    if matches.empty:
        return f"No movie found matching '{title}'"

    idx = matches.index[0]
    print(f"Movies similar to: {movies.loc[idx, 'title']}  [{movies.loc[idx, 'genres']}]\n")

    scores = sorted(enumerate(sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    result = movies.iloc[[i for i, _ in scores]][["title", "genres"]].copy()
    result["similarity"] = [round(s, 3) for _, s in scores]
    return result

similar_movies("Toy Story")

Movies similar to: Toy Story (1995)  [Adventure|Animation|Children|Comedy|Fantasy]



,title,genres,similarity
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0


In [8]:
def similar_movies(title, n=10):
    matches = movies[movies["title"].str.contains(title, case=False, regex=False)]
    if matches.empty:
        return f"No movie found matching '{title}'"

    idx = matches.index[0]
    print(f"Movies similar to: {movies.loc[idx, 'title']}  [{movies.loc[idx, 'genres']}]\n")

    scores = sorted(enumerate(sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    result = movies.iloc[[i for i, _ in scores]][["title", "genres"]].copy()
    result["similarity"] = [round(s, 3) for _, s in scores]
    return result

similar_movies("Batman")

Movies similar to: Batman Forever (1995)  [Action|Adventure|Comedy|Crime]



,title,genres,similarity
126,Batman Forever (1995),Action|Adventure|Comedy|Crime,1.000
3511,"It's a Mad, Mad, Mad, Mad World (1963)",Action|Adventure|Comedy|Crime,1.000
4071,I Spy (2002),Action|Adventure|Comedy|Crime,1.000
4726,Nothing to Lose (1997),Action|Adventure|Comedy|Crime,1.000
6274,Crime Busters (1977),Action|Adventure|Comedy|Crime,1.000
8636,Kingsman: The Secret Service (2015),Action|Adventure|Comedy|Crime,1.000
4005,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,0.949
4681,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,0.949
8027,"Man with the Iron Fists, The (2012)",Action|Adventure|Crime,0.934
8929,Spectre (2015),Action|Adventure|Crime,0.934


In [9]:
def similar_movies_v2(title, n=10, min_ratings=20):
    matches = movies[movies["title"].str.contains(title, case=False, regex=False)]
    if matches.empty:
        return f"No movie found matching '{title}'"

    idx = matches.index[0]
    df = movies[["movieId", "title", "genres"]].copy()
    df["similarity"] = sim[idx]
    df = df.merge(stats, on="movieId", how="left")

    df = df[(df["n_ratings"] >= min_ratings) & (df.index != idx)]
    df["score"] = df["similarity"] * 0.7 + (df["avg_rating"] / 5) * 0.3
    return df.sort_values("score", ascending=False)[["title", "genres", "similarity", "avg_rating", "n_ratings"]].head(n)

similar_movies_v2("Toy Story")

,title,genres,similarity,avg_rating,n_ratings
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.871212,132.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.860825,97.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.716216,37.0
8900,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.970795,3.813953,43.0
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.244444,45.0
8357,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fan...,0.939012,3.870968,31.0
3194,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.933286,3.867647,170.0
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.000000,3.023810,21.0
4360,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.875626,3.960993,141.0
3745,Ice Age (2002),Adventure|Animation|Children|Comedy,0.875626,3.688235,85.0


In [10]:
import numpy as np

user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating")
print(user_item.shape)   # about (610, 9724)
user_item.iloc[:5, :5]

(610, 9724)


movieId,1,2,3,4,5
userId,,,,,
1,4.0,NaN,4.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN


In [11]:
user_means = user_item.mean(axis=1)
centered = user_item.sub(user_means, axis=0).fillna(0)   # unrated = 0 = "neutral"

user_sim = pd.DataFrame(
    cosine_similarity(centered),
    index=user_item.index,
    columns=user_item.index
)
user_sim.iloc[:5, :5]

userId,1,2,3,4,5
userId,,,,,
1,1.000000,0.001265,0.000553,0.048419,0.021847
2,0.001265,1.000000,0.000000,-0.017164,0.021796
3,0.000553,0.000000,1.000000,-0.011260,-0.031539
4,0.048419,-0.017164,-0.011260,1.000000,-0.029620
5,0.021847,0.021796,-0.031539,-0.029620,1.000000


In [12]:
def recommend_for_user(user_id, n=10, k=30, min_neighbors=3):
    # 1. Find the k most similar users
    sims = user_sim[user_id].drop(user_id).nlargest(k)

    # 2. Weighted average of their (centered) ratings, only counting neighbors who rated the movie
    rated = user_item.loc[sims.index].notna()
    numerator = centered.loc[sims.index].mul(sims, axis=0).sum()
    denominator = rated.mul(sims.abs(), axis=0).sum().replace(0, np.nan)

    # 3. Predicted rating = user's own average + neighbors' opinion
    pred = user_means[user_id] + numerator / denominator

    # 4. Keep only unseen movies that enough neighbors have rated
    unseen = user_item.loc[user_id].isna()
    enough = rated.sum() >= min_neighbors
    pred = pred[unseen & enough].dropna().nlargest(n)

    out = pred.rename("predicted_rating").reset_index().merge(movies, on="movieId")
    return out[["title", "genres", "predicted_rating"]].round(2)

recommend_for_user(1)

,title,genres,predicted_rating
0,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,5.91
1,Rosencrantz and Guildenstern Are Dead (1990),Comedy|Drama,5.89
2,"Crow, The (1994)",Action|Crime|Fantasy|Thriller,5.80
3,"Bridge on the River Kwai, The (1957)",Adventure|Drama|War,5.74
4,Princess Mononoke (Mononoke-hime) (1997),Action|Adventure|Animation|Drama|Fantasy,5.74
5,Defending Your Life (1991),Comedy|Drama|Fantasy|Romance,5.73
6,Empire of the Sun (1987),Action|Adventure|Drama|War,5.69
7,"Godfather, The (1972)",Crime|Drama,5.66
8,"Shawshank Redemption, The (1994)",Crime|Drama,5.63
9,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama,5.60


In [13]:
def user_favorites(user_id, n=8):
    r = ratings[ratings["userId"] == user_id].merge(movies, on="movieId")
    return r.sort_values("rating", ascending=False)[["title", "rating"]].head(n)

user_favorites(1)

,title,rating
3,Seven (a.k.a. Se7en) (1995),5.0
4,"Usual Suspects, The (1995)",5.0
6,Bottle Rocket (1996),5.0
13,Dumb & Dumber (Dumb and Dumber) (1994),5.0
11,Billy Madison (1995),5.0
10,Desperado (1995),5.0
9,Canadian Bacon (1995),5.0
8,Rob Roy (1995),5.0


In [14]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size=0.2, random_state=42)
print(len(train), len(test))

80668 20168


In [15]:
def build_cf(train, k=30):
    ui = train.pivot_table(index="userId", columns="movieId", values="rating")
    means = ui.mean(axis=1)
    cen = ui.sub(means, axis=0).fillna(0)

    S = cosine_similarity(cen)
    np.fill_diagonal(S, 0)
    # keep only the k most similar users per row, and only positive similarities
    drop = np.argpartition(-S, k, axis=1)[:, k:]
    np.put_along_axis(S, drop, 0, axis=1)
    S = np.clip(S, 0, None)

    rated = ui.notna().values.astype(float)
    num = S @ cen.values
    den = S @ rated
    delta = np.divide(num, den, out=np.zeros_like(num), where=den > 0)

    pred = pd.DataFrame(means.values[:, None] + delta, index=ui.index, columns=ui.columns)
    support = pd.DataFrame((S > 0).astype(float) @ rated, index=ui.index, columns=ui.columns)
    return pred, support

pred, support = build_cf(train, k=30)
pred.shape

(610, 8983)

In [16]:
global_mean = train["rating"].mean()
movie_mean = train.groupby("movieId")["rating"].mean()

def rmse(pred_values):
    return np.sqrt(np.mean((test["rating"].values - np.array(pred_values)) ** 2))

# Baseline 1: always predict the global average
p_global = [global_mean] * len(test)

# Baseline 2: predict each movie's average rating
p_movie = [movie_mean.get(m, global_mean) for m in test["movieId"]]

# Collaborative filtering
p_cf = []
for u, m in zip(test["userId"], test["movieId"]):
    if u in pred.index and m in pred.columns:
        p_cf.append(float(np.clip(pred.at[u, m], 0.5, 5)))
    else:
        p_cf.append(global_mean)      # cold-start fallback

print(f"Global mean : {rmse(p_global):.4f}")
print(f"Movie mean  : {rmse(p_movie):.4f}")
print(f"Collaborative: {rmse(p_cf):.4f}")

Global mean : 1.0488
Movie mean  : 0.9827
Collaborative: 0.9331


In [17]:
relevant = test[test["rating"] >= 4].groupby("userId")["movieId"].apply(set)
seen = train.groupby("userId")["movieId"].apply(set)

tr_stats = train.groupby("movieId")["rating"].agg(avg="mean", n="count")
pop_rank = tr_stats[tr_stats.n >= 50].sort_values("avg", ascending=False).index.tolist()

def rec_popular(u, K):
    s = seen[u]
    return [m for m in pop_rank if m not in s][:K]

def rec_cf(u, K, min_support=3):
    row = pred.loc[u].where(support.loc[u] >= min_support).dropna()
    row = row[~row.index.isin(seen[u])]
    return row.nlargest(K).index.tolist()

def evaluate_topk(rec_fn, K=10):
    precs, recs = [], []
    for u, rel in relevant.items():
        if u not in seen.index:
            continue
        hits = len(set(rec_fn(u, K)) & rel)
        precs.append(hits / K)
        recs.append(hits / len(rel))
    return np.mean(precs), np.mean(recs)

for name, fn in [("Popularity", rec_popular), ("Collaborative", rec_cf)]:
    p, r = evaluate_topk(fn, K=10)
    print(f"{name:15s} Precision@10 = {p:.3f}   Recall@10 = {r:.3f}")

Popularity      Precision@10 = 0.072   Recall@10 = 0.049
Collaborative   Precision@10 = 0.043   Recall@10 = 0.041


In [18]:
for k in [10, 30, 60, 100]:
    pred, support = build_cf(train, k=k)
    p_cf = [float(np.clip(pred.at[u, m], 0.5, 5)) if (u in pred.index and m in pred.columns) else global_mean
            for u, m in zip(test["userId"], test["movieId"])]
    p, r = evaluate_topk(rec_cf, K=10)
    print(f"k={k:3d}  RMSE={rmse(p_cf):.4f}  Precision@10={p:.3f}  Recall@10={r:.3f}")

k= 10  RMSE=0.9645  Precision@10=0.120  Recall@10=0.121
k= 30  RMSE=0.9331  Precision@10=0.043  Recall@10=0.041
k= 60  RMSE=0.9163  Precision@10=0.014  Recall@10=0.013
k=100  RMSE=0.9042  Precision@10=0.006  Recall@10=0.004


In [19]:
import joblib, os
from scipy import sparse

os.makedirs("../backend/artifacts", exist_ok=True)

BEST_K = 30   # put the best k from Cell 17 here

# Final model: trained on ALL ratings (the train/test split was only for evaluation)
pred_full, support_full = build_cf(ratings, k=BEST_K)
seen_full = ratings.groupby("userId")["movieId"].apply(set).to_dict()

joblib.dump({
    "pred": pred_full.astype("float32"),
    "support": support_full.astype("int16"),
    "seen": seen_full,
    "movies": movies[["movieId", "title", "genres"]],
    "stats": stats,                      # movieId, avg_rating, n_ratings
}, "../backend/artifacts/model.joblib", compress=3)

sparse.save_npz("../backend/artifacts/genre_matrix.npz", genre_matrix)
print("saved")

saved


In [20]:
ui = ratings.pivot_table(index="userId", columns="movieId", values="rating")
means = ui.mean(axis=1)
cen = ui.sub(means, axis=0).fillna(0)

joblib.dump({
    "centered": cen.values.astype("float32"),
    "rated": ui.notna().values.astype("float32"),
    "columns": ui.columns.tolist(),
}, "../backend/artifacts/cf_matrix.joblib", compress=3)
print("saved", cen.shape)

saved (610, 9724)
